In [ ]:

import torch
from open_spiel.python import rl_environment
from open_spiel.python.pytorch import dqn

# Environment
env = rl_environment.Environment("pentago")
num_actions = env.action_spec()["num_actions"]
state_size = env.observation_spec()["info_state"][0]

In [ ]:
agents = [
    dqn.DQN(
        player_id=i,
        state_representation_size=state_size,
        num_actions=num_actions,
        hidden_layers_sizes=[256, 256],
        replay_buffer_capacity=100_000,
        batch_size=128,
        learning_rate=1e-3,
        update_target_network_every=500,
        learn_every=10,
        discount_factor=0.99,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_duration=50_000,
    )
    for i in range(2)
]

# Training loop
num_episodes = 20_000

for ep in range(num_episodes):
    time_step = env.reset()

    while not time_step.last():
        current_player = time_step.observations["current_player"]
        agent_output = agents[current_player].step(time_step)
        time_step = env.step([agent_output.action])

    # Let both agents observe the terminal state
    for agent in agents:
        agent.step(time_step)

    # Logging
    if ep % 500 == 0:
        print(f"Episode {ep}/{num_episodes} | "
              f"P0 loss: {agents[0].loss} | "
              f"P1 loss: {agents[1].loss} | ")

# Save the trained agents
torch.save(agents[0]._q_network.state_dict(), "agent0.pt")
torch.save(agents[1]._q_network.state_dict(), "agent1.pt")
print("Training complete!")

Episode 0/20000 | P0 loss: None | P1 loss: None | 
Episode 500/20000 | P0 loss: 0.17843368649482727 | P1 loss: 0.16925212740898132 | 
Episode 1000/20000 | P0 loss: 0.0765342116355896 | P1 loss: 0.07970569282770157 | 
Episode 1500/20000 | P0 loss: 0.11845698952674866 | P1 loss: 0.09224559366703033 | 
Episode 2000/20000 | P0 loss: 0.10308422148227692 | P1 loss: 0.12114106863737106 | 
Episode 2500/20000 | P0 loss: 0.09296473115682602 | P1 loss: 0.1535455584526062 | 
Episode 3000/20000 | P0 loss: 0.055810533463954926 | P1 loss: 0.11739137768745422 | 
Episode 3500/20000 | P0 loss: 0.07551909983158112 | P1 loss: 0.12596218287944794 | 
Episode 4000/20000 | P0 loss: 0.10682070255279541 | P1 loss: 0.10440780222415924 | 
Episode 4500/20000 | P0 loss: 0.11426521837711334 | P1 loss: 0.047957561910152435 | 
Episode 5000/20000 | P0 loss: 0.07952475547790527 | P1 loss: 0.0602685920894146 | 
Episode 5500/20000 | P0 loss: 0.12286542356014252 | P1 loss: 0.0627979263663292 | 
Episode 6000/20000 | P0 loss

In [61]:
from open_spiel.python.algorithms import random_agent

def eval_vs_random(trained_agent, num_episodes=1000, agent_player_id=0):
    rand_agent = random_agent.RandomAgent(
        player_id=1 - agent_player_id,
        num_actions=num_actions
    )
    
    wins, losses, draws = 0, 0, 0

    for ep in range(num_episodes):
        time_step = env.reset()

        while not time_step.last():
            current_player = time_step.observations["current_player"]
            if current_player == agent_player_id:
                action = trained_agent.step(time_step, is_evaluation=True).action
            else:
                action = rand_agent.step(time_step).action
            time_step = env.step([action])

        # Check result
        returns = time_step.observations["info_state"]  # not quite right, use env
        final_returns = env._state.returns()
        if final_returns[agent_player_id] > 0:
            wins += 1
        elif final_returns[agent_player_id] < 0:
            losses += 1
        else:
            draws += 1

        if (ep + 1) % 100 == 0:
            print(f"Episode {ep+1}: W={wins} L={losses} D={draws} | "
                  f"Winrate: {wins/(ep+1):.2%}")

    print(f"\nFinal over {num_episodes} games:")
    print(f"  Wins:   {wins}  ({wins/num_episodes:.2%})")
    print(f"  Losses: {losses}  ({losses/num_episodes:.2%})")
    print(f"  Draws:  {draws}  ({draws/num_episodes:.2%})")

# Evaluate after training
eval_vs_random(agents[0], num_episodes=500, agent_player_id=0)

Episode 100: W=58 L=38 D=4 | Winrate: 58.00%
Episode 200: W=105 L=84 D=11 | Winrate: 52.50%
Episode 300: W=165 L=120 D=15 | Winrate: 55.00%
Episode 400: W=214 L=161 D=25 | Winrate: 53.50%
Episode 500: W=266 L=198 D=36 | Winrate: 53.20%

Final over 500 games:
  Wins:   266  (53.20%)
  Losses: 198  (39.60%)
  Draws:  36  (7.20%)
